# Druggability Axis Analysis

### Goal
Identify existing therapeutic compounds that target the broadly and strictly conserved pan-cancer metastatic metabolic genes.

### Purpose
To cross-reference the highly conserved metastatic metabolic signature against DGIdb to find druggable vulnerabilities. This helps prioritize targets for potential drug repurposing.

### Interpretation
- **Druggable Targets:** Genes with known drug interactions.
- **Clinical Relevance:** Overlap with FDA-approved drugs indicates immediate translational potential for treating metastasis.

### Inputs / Parameters
*Explicitly documented for traceability and reproducibility.*

**Configuration Files:**
- `pan_cancer_config.py`
- `pan_cancer_config.py`


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

%load_ext autoreload
%autoreload 2

from IPython.display import display, HTML
from dynamic_genes import get_dynamic_genes
TARGET_GENES = get_dynamic_genes('.')
from pan_cancer_config import ANALYSIS_SUFFIX
OUTPUT_DIR = '../output/druggability'
OUTPUT_BASENAME = f'druggability_axis{ANALYSIS_SUFFIX}'
from query_dbs import compile_drug_databases
from query_depmap import analyze_depmap_synergy
from query_advanced_analysis import query_tractability, query_string_ppi

In [2]:
# ==========================================
# ⚙️ USER PARAMETERS (Export Options)
# ==========================================
# Full Notebook HTML Report Export Toggle:
# - True  -> Automatically exports the entire notebook (Markdown, Code, Tables, Figures) to styled HTML
# - False -> Disables automatic HTML export
SAVE_AS_HTML = True  # DEFAULT: False. Change to True to export the entire notebook!

print(f"Analyzing Axis: {TARGET_GENES}")
print(f"Output Base: {OUTPUT_BASENAME}")

Analyzing Axis: ['SELL', 'LGSN', 'CNDP1', 'ACACA', 'S1PR1', 'SPTLC1', 'AMN', 'ANXA1', 'CD247', 'CHRNA6', 'NMUR1', 'TNFRSF21', 'DBT', 'IL17RA', 'IL1RAP', 'DGAT2', 'ITGB2', 'CXADR', 'SULT1A1', 'NR1D2', 'FCER2', 'SIRPB1', 'TGFBR2', 'SLC2A6', 'AMDHD1', 'EPHB1', 'KCNN2', 'IL1R1', 'P2RY14', 'GCNT1', 'GALNT1', 'LTB4R', 'TRPM7', 'SRP54', 'CD48', 'AUH', 'CSF3R', 'GCNT4', 'LILRB1', 'GPC5', 'SLC16A7', 'HLA-C', 'VTN', 'CNR2', 'GBE1', 'EPOR', 'ADRA2B', 'RXRA', 'CD1D', 'PLCB1', 'THRB', 'PLXNA4', 'IL7R', 'ITGA4', 'DGKD', 'ASGR1', 'CYP19A1', 'SCD5', 'ITGAM', 'TSHR', 'CERT1', 'ACKR3', 'PDE3B', 'OCLN', 'GABRG3', 'C1GALT1', 'GLS', 'TRPM8', 'FZD6', 'P2RY8', 'ADAM10', 'ACVR1B', 'CYSLTR1', 'JAML', 'SLC11A2', 'CD36', 'CD79A', 'CELSR1', 'TLR10', 'FADS2', 'CR2', 'CCR9', 'SLC46A1', 'ACSL4', 'CYP2E1', 'ADORA3', 'GCLM', 'CD3G', 'FCN1', 'IL13RA2', 'CXCR1', 'KLRG1', 'CUL5', 'ABCB5', 'SPN', 'GJA5', 'CD244', 'CDH23', 'GRM1', 'SLC22A1', 'SLC16A6', 'NCOA3', 'PLCL1', 'ESRRG', 'CYSLTR2', 'DGKA', 'NPFFR2', 'MTMR1', 'ITGAL

## 2. Cross-reference Genes against Drug Databases

**Purpose**: Identifies if the genes in the hypothesized axis already have FDA-approved drugs or compounds in clinical trials.  
**Interpretation**: If a gene is druggable, we can rapidly test repurposing existing drugs. If multiple genes in the axis are druggable, it opens the door to combination therapy.

In [3]:
df_drugs = compile_drug_databases(TARGET_GENES)

if not df_drugs.empty:
    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_BASENAME}_drug_targets.csv")
    df_drugs.to_csv(csv_path, index=False)
    print(f"Saved drug targets to {csv_path}")
    display(df_drugs.head(15))
else:
    print("No drug interactions found.")

[DGIdb] Querying interactions for 126 genes...
[DGIdb] Found 1599 interactions.
[OpenTargets] Resolving gene symbols to Ensembl IDs for 126 genes...
[OpenTargets] Resolved IDs: {'SELL': 'ENSG00000188404', 'LGSN': 'ENSG00000146166', 'CNDP1': 'ENSG00000150656', 'ACACA': 'ENSG00000278540', 'S1PR1': 'ENSG00000170989', 'SPTLC1': 'ENSG00000090054', 'AMN': 'ENSG00000166126', 'ANXA1': 'ENSG00000135046', 'CD247': 'ENSG00000198821', 'CHRNA6': 'ENSG00000147434', 'NMUR1': 'ENSG00000171596', 'TNFRSF21': 'ENSG00000146072', 'DBT': 'ENSG00000137992', 'IL17RA': 'ENSG00000177663', 'IL1RAP': 'ENSG00000196083', 'DGAT2': 'ENSG00000062282', 'ITGB2': 'ENSG00000160255', 'CXADR': 'ENSG00000154639', 'SULT1A1': 'ENSG00000196502', 'NR1D2': 'ENSG00000174738', 'FCER2': 'ENSG00000104921', 'SIRPB1': 'ENSG00000101307', 'TGFBR2': 'ENSG00000163513', 'SLC2A6': 'ENSG00000160326', 'AMDHD1': 'ENSG00000139344', 'EPHB1': 'ENSG00000154928', 'KCNN2': 'ENSG00000080709', 'IL1R1': 'ENSG00000115594', 'P2RY14': 'ENSG00000174944', 'G

,Database,Target_Gene,Drug_Name,Interaction_Type,Sources,Approval_Status,Indication_Disease
0,DGIdb,SLC22A1,MERCAPTOPURINE,Targeted,DGIdb GraphQL,N/A,NaN
1,DGIdb,SLC22A1,OLANZAPINE,Targeted,DGIdb GraphQL,N/A,NaN
2,DGIdb,SLC22A1,FENOTEROL,Targeted,DGIdb GraphQL,N/A,NaN
3,DGIdb,SLC22A1,TETRAMETHYLAMMONIUM,Targeted,DGIdb GraphQL,N/A,NaN
4,DGIdb,SLC22A1,AMANTADINE HYDROCHLORIDE,Targeted,DGIdb GraphQL,N/A,NaN
5,DGIdb,SLC22A1,SUMATRIPTAN,Targeted,DGIdb GraphQL,N/A,NaN
6,DGIdb,SLC22A1,ELTROMBOPAG,Targeted,DGIdb GraphQL,N/A,NaN
7,DGIdb,SLC22A1,SORAFENIB,Targeted,DGIdb GraphQL,N/A,NaN
8,DGIdb,SLC22A1,RANITIDINE HYDROCHLORIDE,Targeted,DGIdb GraphQL,N/A,NaN
9,DGIdb,SLC22A1,CYCLOGUANIL,Targeted,DGIdb GraphQL,N/A,NaN


## 3. Target Tractability Prediction (Open Targets)

**Purpose**: For targets lacking known drugs, we query computational models to predict if they are structurally "druggable" by small molecules or antibodies.  
**Interpretation**: High tractability scores indicate that a gene is a biologically viable target for *de novo* drug discovery, even if no drugs currently exist.

In [4]:
df_trac = query_tractability(TARGET_GENES)
if not df_trac.empty:
    display(df_trac)
else:
    print("No tractability data found.")

[Tractability] Querying Open Targets for tractability of ['SELL', 'LGSN', 'CNDP1', 'ACACA', 'S1PR1', 'SPTLC1', 'AMN', 'ANXA1', 'CD247', 'CHRNA6', 'NMUR1', 'TNFRSF21', 'DBT', 'IL17RA', 'IL1RAP', 'DGAT2', 'ITGB2', 'CXADR', 'SULT1A1', 'NR1D2', 'FCER2', 'SIRPB1', 'TGFBR2', 'SLC2A6', 'AMDHD1', 'EPHB1', 'KCNN2', 'IL1R1', 'P2RY14', 'GCNT1', 'GALNT1', 'LTB4R', 'TRPM7', 'SRP54', 'CD48', 'AUH', 'CSF3R', 'GCNT4', 'LILRB1', 'GPC5', 'SLC16A7', 'HLA-C', 'VTN', 'CNR2', 'GBE1', 'EPOR', 'ADRA2B', 'RXRA', 'CD1D', 'PLCB1', 'THRB', 'PLXNA4', 'IL7R', 'ITGA4', 'DGKD', 'ASGR1', 'CYP19A1', 'SCD5', 'ITGAM', 'TSHR', 'CERT1', 'ACKR3', 'PDE3B', 'OCLN', 'GABRG3', 'C1GALT1', 'GLS', 'TRPM8', 'FZD6', 'P2RY8', 'ADAM10', 'ACVR1B', 'CYSLTR1', 'JAML', 'SLC11A2', 'CD36', 'CD79A', 'CELSR1', 'TLR10', 'FADS2', 'CR2', 'CCR9', 'SLC46A1', 'ACSL4', 'CYP2E1', 'ADORA3', 'GCLM', 'CD3G', 'FCN1', 'IL13RA2', 'CXCR1', 'KLRG1', 'CUL5', 'ABCB5', 'SPN', 'GJA5', 'CD244', 'CDH23', 'GRM1', 'SLC22A1', 'SLC16A6', 'NCOA3', 'PLCL1', 'ESRRG', 'CY

,Target_Gene,Small_Molecule_Tractable,Antibody_Tractable,Other_Modalities_Tractable,Source
0,SELL,False,False,False,Cache
1,LGSN,False,False,False,Cache
2,CNDP1,False,False,False,Cache
3,ACACA,False,False,False,Cache
4,S1PR1,False,False,False,Cache
...,...,...,...,...,...
121,SLC15A1,False,False,False,Cache
122,ESR1,False,False,False,Cache
123,DGKG,False,False,False,Cache
124,AGER,False,False,False,Cache


## 4. Protein-Protein Interaction (STRING Database)

**Purpose**: Queries the STRING network to check if there is physical or functional evidence that these targets interact with each other.  
**Interpretation**: High combined scores (>0.4) strongly support the "axis" hypothesis, indicating these genes operate in the same complex or pathway rather than in isolation.

In [5]:
df_ppi = query_string_ppi(TARGET_GENES)
if not df_ppi.empty:
    display(df_ppi)
else:
    print("No PPI network connections found between these genes.")

[STRING PPI] Querying network for: ['SELL', 'LGSN', 'CNDP1', 'ACACA', 'S1PR1', 'SPTLC1', 'AMN', 'ANXA1', 'CD247', 'CHRNA6', 'NMUR1', 'TNFRSF21', 'DBT', 'IL17RA', 'IL1RAP', 'DGAT2', 'ITGB2', 'CXADR', 'SULT1A1', 'NR1D2', 'FCER2', 'SIRPB1', 'TGFBR2', 'SLC2A6', 'AMDHD1', 'EPHB1', 'KCNN2', 'IL1R1', 'P2RY14', 'GCNT1', 'GALNT1', 'LTB4R', 'TRPM7', 'SRP54', 'CD48', 'AUH', 'CSF3R', 'GCNT4', 'LILRB1', 'GPC5', 'SLC16A7', 'HLA-C', 'VTN', 'CNR2', 'GBE1', 'EPOR', 'ADRA2B', 'RXRA', 'CD1D', 'PLCB1', 'THRB', 'PLXNA4', 'IL7R', 'ITGA4', 'DGKD', 'ASGR1', 'CYP19A1', 'SCD5', 'ITGAM', 'TSHR', 'CERT1', 'ACKR3', 'PDE3B', 'OCLN', 'GABRG3', 'C1GALT1', 'GLS', 'TRPM8', 'FZD6', 'P2RY8', 'ADAM10', 'ACVR1B', 'CYSLTR1', 'JAML', 'SLC11A2', 'CD36', 'CD79A', 'CELSR1', 'TLR10', 'FADS2', 'CR2', 'CCR9', 'SLC46A1', 'ACSL4', 'CYP2E1', 'ADORA3', 'GCLM', 'CD3G', 'FCN1', 'IL13RA2', 'CXCR1', 'KLRG1', 'CUL5', 'ABCB5', 'SPN', 'GJA5', 'CD244', 'CDH23', 'GRM1', 'SLC22A1', 'SLC16A6', 'NCOA3', 'PLCL1', 'ESRRG', 'CYSLTR2', 'DGKA', 'NPFFR

,Gene_A,Gene_B,Combined_Score,Experimental_Score,Database_Score,Source
0,CD79A,CD48,0.403,0.000,0.0,Cache
1,CD79A,CD247,0.418,0.000,0.0,Cache
2,CD79A,ITGB2,0.421,0.000,0.0,Cache
3,CD79A,ESR1,0.463,0.000,0.0,Cache
4,CD79A,IL7R,0.505,0.000,0.0,Cache
...,...,...,...,...,...,...
290,ITGAM,SPN,0.749,0.000,0.0,Cache
291,ITGAM,SELL,0.929,0.000,0.0,Cache
292,SELL,CD1D,0.624,0.000,0.0,Cache
293,SELL,SPN,0.791,0.099,0.0,Cache


## 5. Synergistic Cell Death (DepMap Co-dependency)

**Purpose**: Uses Cancer Dependency Map (DepMap) CRISPR knockout data to calculate the correlation of gene effects across hundreds of cancer cell lines.  
**Interpretation**: 
- **Positive correlation (> 0.3)**: Suggests genes are in the same functional pathway; knocking out either has a similar effect.
- **Negative correlation**: Suggests potential synthetic lethality or compensatory mechanisms.

In [6]:
corr_matrix = analyze_depmap_synergy(TARGET_GENES)

if corr_matrix is not None:
    display(corr_matrix)
else:
    print("Skipping correlation visualization.")

[DepMap] Loading DepMap CRISPR data from /Users/sakuramaezono/Library/CloudStorage/OneDrive-YokohamaCityUniversity/Personal/05_Python_repositories/metabConnectomeDB/input/CRISPRGeneEffect.csv...
[DepMap] Found columns for: ['ABCB5', 'ACACA', 'ACKR3', 'ACSL4', 'ACVR1B', 'ACVR2B', 'ADAM10', 'ADORA3', 'ADRA2B', 'AGER', 'AMDHD1', 'AMN', 'ANXA1', 'ASGR1', 'AUH', 'C1GALT1', 'CCR9', 'CD1D', 'CD244', 'CD247', 'CD36', 'CD3G', 'CD46', 'CD48', 'CD79A', 'CDH23', 'CELSR1', 'CERT1', 'CHRNA6', 'CNDP1', 'CNR2', 'CR2', 'CSF3R', 'CUL5', 'CXADR', 'CXCR1', 'CYP19A1', 'CYP1B1', 'CYP2E1', 'CYSLTR1', 'CYSLTR2', 'DBT', 'DGAT2', 'DGKA', 'DGKD', 'DGKG', 'EPHB1', 'EPOR', 'ERAP1', 'ESR1', 'ESRRG', 'FADS2', 'FCER2', 'FCN1', 'FZD6', 'GABRG3', 'GALNT1', 'GBE1', 'GCLM', 'GCNT1', 'GCNT4', 'GFRA2', 'GJA5', 'GLS', 'GPC5', 'GRM1', 'HK3', 'HLA-C', 'IFNLR1', 'IL13RA2', 'IL17RA', 'IL1R1', 'IL1RAP', 'IL7R', 'ITGA4', 'ITGAL', 'ITGAM', 'ITGB2', 'JAML', 'KCNN2', 'KLRG1', 'LGSN', 'LILRB1', 'LTB4R', 'MTMR1', 'NCOA3', 'NMUR1', 'NP

,ABCB5,ACACA,ACKR3,ACSL4,ACVR1B,ACVR2B,ADAM10,ADORA3,ADRA2B,AGER,AMDHD1,AMN,ANXA1,ASGR1,AUH,C1GALT1,CCR9,CD1D,CD244,CD247,CD36,CD3G,CD46,CD48,CD79A,CDH23,CELSR1,CERT1,CHRNA6,CNDP1,CNR2,CR2,CSF3R,CUL5,CXADR,CXCR1,CYP19A1,CYP1B1,CYP2E1,CYSLTR1,CYSLTR2,DBT,DGAT2,DGKA,DGKD,DGKG,EPHB1,EPOR,ERAP1,ESR1,ESRRG,FADS2,FCER2,FCN1,FZD6,GABRG3,GALNT1,GBE1,GCLM,GCNT1,GCNT4,GFRA2,GJA5,GLS,GPC5,GRM1,HK3,HLA-C,IFNLR1,IL13RA2,IL17RA,IL1R1,IL1RAP,IL7R,ITGA4,ITGAL,ITGAM,ITGB2,JAML,KCNN2,KLRG1,LGSN,LILRB1,LTB4R,MTMR1,NCOA3,NMUR1,NPFFR2,NR1D2,NR2F1,OCLN,OGDH,P2RY14,P2RY8,PDE3B,PGS1,PLCB1,PLCL1,PLXNA4,RXRA,S1PR1,SCD5,SELL,SIRPB1,SLC11A2,SLC15A1,SLC16A6,SLC16A7,SLC1A2,SLC22A1,SLC2A12,SLC2A6,SLC46A1,SLC6A3,SPN,SPTLC1,SRP54,SULT1A1,TGFBR2,THRB,TLR10,TNFRSF21,TRPM7,TRPM8,TSHR,VTN
ABCB5,1.000000,-0.042607,-0.066138,0.029534,0.026495,0.004620,0.017556,0.104035,-0.088403,0.014589,-0.008695,-0.170281,-0.101194,-0.078671,0.101801,-0.042942,-0.117015,0.055362,0.149121,-0.060456,0.097802,0.006550,-0.005264,0.179712,-0.095651,-0.085124,-0.140205,-0.021066,-0.026614,-0.063903,-0.044170,0.111913,0.041775,0.097163,0.051955,0.023766,-0.052877,-0.128301,0.004042,0.250487,0.083233,0.072621,-0.115298,-0.058446,-0.102031,-0.008395,-0.033050,-0.064790,0.102037,-0.066699,-0.029353,0.114274,0.108158,-0.096313,-0.014521,0.010281,-0.002602,0.062133,-0.011060,0.049709,0.103011,0.000655,0.049291,-0.044548,2.275681e-02,-0.133370,-0.083829,-0.016125,-0.075152,0.241734,-0.063941,0.022943,0.149800,-0.032103,-0.017299,-0.068575,-0.055792,-0.033002,0.012821,0.019565,0.140244,0.037916,-0.046090,-0.024458,0.095315,-0.019873,-0.051331,0.017955,0.048472,0.066638,0.019286,-0.014159,-0.101131,0.117080,0.103026,0.047922,0.027854,-0.045552,-0.112731,-0.089542,-0.013150,0.041483,0.085133,-0.072105,-0.008149,-0.033678,0.021945,0.142619,0.037776,-0.111980,-0.045935,-0.087366,0.119058,-0.026823,0.045007,0.123893,-0.001475,-0.163121,0.006126,-0.013681,0.167791,-0.043282,0.000775,0.051504,0.045947,0.088284
ACACA,-0.042607,1.000000,-0.009671,-0.084003,0.016012,0.013597,0.009638,-0.100611,0.039147,-0.049982,-0.071301,-0.004200,0.046012,-0.015343,0.001372,-0.010221,0.060736,-0.043321,0.006662,0.085659,-0.020203,-0.030521,0.047671,-0.021388,0.080582,-0.041958,0.057685,0.161293,-0.055508,0.066915,-0.031565,0.037099,-0.083408,-0.043949,0.123770,0.017621,0.018427,-0.028110,0.014133,-0.087484,-0.026553,0.010960,0.033956,-0.005301,-0.033261,0.012371,-0.049179,0.013162,0.078084,-0.013196,-0.035626,-0.049106,0.031118,0.085494,0.064281,0.048301,-0.036812,0.021861,0.030076,0.059902,0.032313,0.005873,0.050481,0.075699,7.884744e-03,0.022327,0.014881,-0.051488,-0.031664,-0.014632,0.026949,0.058255,0.020519,0.042345,-0.063161,0.018826,-0.057509,-0.031830,-0.028425,0.000597,0.047192,0.026929,0.030888,-0.071722,0.031354,0.020739,0.008390,-0.031801,-0.065899,0.022047,-0.023763,0.063169,0.044172,-0.106898,-0.028844,0.156608,0.007522,0.008876,0.011975,0.062075,0.029316,-0.078783,-0.019887,0.023490,0.053764,-0.016947,-0.010404,0.026303,-0.023714,0.002501,0.006878,-0.019060,0.108666,-0.016622,-0.026257,0.009435,-0.075849,0.044780,0.016313,0.018583,-0.000681,0.000015,0.054123,-0.027519,0.029868,0.031333
ACKR3,-0.066138,-0.009671,1.000000,0.034255,0.019095,0.127903,-0.151822,-0.017412,-0.006842,0.033522,-0.017872,0.111955,-0.030224,0.039015,-0.071782,-0.056893,0.065651,-0.060820,-0.101866,-0.008916,0.093535,0.008630,-0.004546,-0.081658,0.086189,0.055503,0.146014,-0.030900,-0.137684,0.104732,0.047826,-0.065190,0.048660,-0.080958,-0.054607,-0.073603,0.008755,0.062119,0.052717,-0.199703,-0.109592,-0.061979,0.018428,0.057426,0.149605,0.055782,0.012508,0.052695,-0.013895,0.118483,0.073382,-0.075231,-0.126970,0.093554,-0.086497,0.080643,0.023897,0.000275,-0.045852,-0.087629,-0.160148,0.062383,0.022867,0.058466,6.238501e-02,0.016510,0.221887,0.096407,0.027585,-0.057505,-0.024757,0.036813,-0.011976,-0.016563,0.046610,-0.034289,0.079459,0.145949,0.026152,0.049340,-0.061699,-0.041115,0.037028,0.011399,-0.005851,0.025800,0.172331,0.

---

## 6. Export Full Notebook Report to HTML

Compiling this entire interactive notebook into a single publication-ready and highly interactive HTML report.

---
## Druggability of Pan-Cancer Conserved Genes
In addition to the specific GLS axis, we also query the DGIdb database for the strictly conserved **strictly conserved pan-cancer genes** (upregulated in all evaluated cancers) and the broadly conserved **broadly conserved genes** (upregulated in $\ge$ N-1 cancers).


In [7]:
import sys
if '..' not in sys.path: sys.path.append('..')
from pan_cancer_config import ANALYSIS_SUFFIX

df_23 = pd.read_csv(os.path.join(OUTPUT_DIR, f'druggable_targets_strictly_conserved{ANALYSIS_SUFFIX}.csv'))
print(f"Total drug interactions for strictly conserved genes: {len(df_23)}")
display(df_23.head(10))

Total drug interactions for strictly conserved genes: 182


,Gene,Drug,Database
0,CD3G,VISILIZUMAB,DGIdb
1,CD3G,ELRANATAMAB,DGIdb
2,CD3G,MUROMONAB-CD3,DGIdb
3,CD3G,TEBENTAFUSP,DGIdb
4,CD3G,EPCORITAMAB,DGIdb
5,CD3G,SOLITOMAB,DGIdb
6,CD3G,TEPLIZUMAB,DGIdb
7,CD3G,MEDI-565,DGIdb
8,CD3G,GLOFITAMAB,DGIdb
9,CD3G,FBT-A05,DGIdb


In [8]:
import sys
if '..' not in sys.path: sys.path.append('..')
from pan_cancer_config import ANALYSIS_SUFFIX

df_181 = pd.read_csv(os.path.join(OUTPUT_DIR, f'druggable_targets_broadly_conserved{ANALYSIS_SUFFIX}.csv'))
print(f"Total drug interactions for broadly conserved genes: {len(df_181)}")
display(df_181.head(10))

Total drug interactions for broadly conserved genes: 182


,Gene,Drug,Database
0,CD3G,VISILIZUMAB,DGIdb
1,CD3G,ELRANATAMAB,DGIdb
2,CD3G,MUROMONAB-CD3,DGIdb
3,CD3G,TEBENTAFUSP,DGIdb
4,CD3G,EPCORITAMAB,DGIdb
5,CD3G,SOLITOMAB,DGIdb
6,CD3G,TEPLIZUMAB,DGIdb
7,CD3G,MEDI-565,DGIdb
8,CD3G,GLOFITAMAB,DGIdb
9,CD3G,FBT-A05,DGIdb
